Version: 02.14.2023

# Capstone Project: Bringing It All Together

In this lab, you will bring together many of the tools and techniques that you have learned throughout this course into a final project. You can choose from many different paths to get to the solution. You could use AWS Managed Services, such as Amazon Comprehend, or use the Amazon SageMaker models. Have fun on whichever path you choose.

### Business scenario

You work for a training organization that recently developed an introductory course about machine learning (ML). The course includes more than 40 videos that cover a broad range of ML topics. You have been asked to create an application that will students can use to quickly locate and view video content by searching for topics and key phrases.

You have downloaded all of the videos to an Amazon Simple Storage Service (Amazon S3) bucket. Your assignment is to produce a dashboard that meets your supervisor’s requirements.

To assist you, all of the previous labs have been provided in this workspace.

## Lab steps

To complete this lab, you will follow these steps:

1. [Viewing the video files](#1.-Viewing-the-video-files)
2. [Transcribing the videos](#2.-Transcribing-the-videos)
3. [Normalizing the text](#3.-Normalizing-the-text)
4. [Extracting key phrases and topics](#4.-Extracting-key-phrases-and-topics)
5. [Creating the dashboard](#5.-Creating-the-dashboard)

## Submitting your work

1. In the lab console, choose **Submit** to record your progress and when prompted, choose **Yes**.

1. If the results don't display after a couple of minutes, return to the top of these instructions and choose **Grades**.

     **Tip**: You can submit your work multiple times. After you change your work, choose **Submit** again. Your last submission is what will be recorded for this lab.

1. To find detailed feedback on your work, choose **Details** followed by **View Submission Report**.

## Useful information

The following cell contains some information that might be useful as you complete this project.

In [1]:
bucket = "c190312a4909453l14790296t1w670353558621-labbucket-zp0ornrldsqf"
job_data_access_role = 'arn:aws:iam::670353558621:role/service-role/c190312a4909453l14790296t1-ComprehendDataAccessRole-HO9gdiW5eAqD'

## 1. Viewing the video files
([Go to top](#Capstone-8:-Bringing-It-All-Together))


The source video files are located in the following shared Amazon Simple Storage Service (Amazon S3) bucket.

In [2]:
!aws s3 ls s3://aws-tc-largeobjects/CUR-TF-200-ACMNLP-1/video/

2021-04-26 20:17:33  410925369 Mod01_Course Overview.mp4
2021-04-26 20:10:02   39576695 Mod02_Intro.mp4
2021-04-26 20:31:23  302994828 Mod02_Sect01.mp4
2021-04-26 20:17:33  416563881 Mod02_Sect02.mp4
2021-04-26 20:17:33  318685583 Mod02_Sect03.mp4
2021-04-26 20:17:33  255877251 Mod02_Sect04.mp4
2021-04-26 20:23:51   99988046 Mod02_Sect05.mp4
2021-04-26 20:24:54   50700224 Mod02_WrapUp.mp4
2021-04-26 20:26:27   60627667 Mod03_Intro.mp4
2021-04-26 20:26:28  272229844 Mod03_Sect01.mp4
2021-04-26 20:27:06  309127124 Mod03_Sect02_part1.mp4
2021-04-26 20:27:06  195635527 Mod03_Sect02_part2.mp4
2021-04-26 20:28:03  123924818 Mod03_Sect02_part3.mp4
2021-04-26 20:31:28  171681915 Mod03_Sect03_part1.mp4
2021-04-26 20:32:07  285200083 Mod03_Sect03_part2.mp4
2021-04-26 20:33:17  105470345 Mod03_Sect03_part3.mp4
2021-04-26 20:35:10  157185651 Mod03_Sect04_part1.mp4
2021-04-26 20:36:27  187435635 Mod03_Sect04_part2.mp4
2021-04-26 20:36:40  280720369 Mod03_Sect04_part3.mp4
2021-04-26 20:40:01  443479

## 2. Transcribing the videos
 ([Go to top](#Capstone-8:-Bringing-It-All-Together))

Use this section to implement your solution to transcribe the videos.

In [3]:
import boto3
import time
import json
import uuid

bucket = "c190312a4909453l14790296t1w301070396965-labbucket-flsgro8xdek1"
job_data_access_role = 'arn:aws:iam::301070396965:role/service-role/c190312a4909453l14790296t1-ComprehendDataAccessRole-gAlehpJYsV3j'

s3 = boto3.client('s3')
transcribe = boto3.client('transcribe')

print("✅ Clients ready")

✅ Clients ready


In [4]:
# Test if Transcribe is now accessible
try:
    transcribe.get_transcription_job(TranscriptionJobName='test-reset')
except transcribe.exceptions.NotFoundException:
    print("✅ Transcribe is accessible! Let's go!")
except Exception as e:
    print(f"❌ Still blocked: {e}")

❌ Still blocked: An error occurred (AccessDeniedException) when calling the GetTranscriptionJob operation: User: arn:aws:sts::670353558621:assumed-role/c190312a4909453l14790296t1w6-SageMakerExecutionRole-O3GoCHvvnLyl/SageMaker is not authorized to perform: transcribe:GetTranscriptionJob on resource: arn:aws:transcribe:us-east-1:670353558621:transcription-job/test-reset because no identity-based policy allows the transcribe:GetTranscriptionJob action


In [5]:
import boto3

bucket = "c190312a4909453l14790296t1w301070396965-labbucket-flsgro8xdek1"
job_data_access_role = 'arn:aws:iam::670353558621:role/service-role/c190312a4909453l14790296t1-ComprehendDataAccessRole-gAlehpJYsV3j'

tests = {
    'Comprehend - DetectKeyPhrases': lambda: boto3.client('comprehend').detect_key_phrases(Text='test sentence', LanguageCode='en'),
    'Comprehend - DetectEntities': lambda: boto3.client('comprehend').detect_entities(Text='test sentence', LanguageCode='en'),
    'Comprehend - StartTopicsJob': lambda: boto3.client('comprehend').start_topics_detection_job(
        InputDataConfig={'S3Uri': f's3://{bucket}/transcripts/', 'InputFormat': 'ONE_DOC_PER_FILE'},
        OutputDataConfig={'S3Uri': f's3://{bucket}/topics/'},
        DataAccessRoleArn=job_data_access_role),
    'S3 - ListObjects': lambda: boto3.client('s3').list_objects_v2(Bucket=bucket),
    'Textract': lambda: boto3.client('textract').detect_document_text(Document={'Bytes': b'test'}),
}

for service, test in tests.items():
    try:
        test()
        print(f"✅ {service}")
    except Exception as e:
        error = str(e)
        if 'AccessDenied' in error or 'not authorized' in error:
            print(f"❌ {service} — BLOCKED")
        else:
            print(f"✅ {service} (accessible)")

❌ Comprehend - DetectKeyPhrases — BLOCKED
❌ Comprehend - DetectEntities — BLOCKED
✅ Comprehend - StartTopicsJob
✅ S3 - ListObjects (accessible)
✅ Textract (accessible)


## 3. Normalizing the text
([Go to top](#Capstone-8:-Bringing-It-All-Together))

Use this section to perform any text normalization steps that are necessary for your solution.

In [8]:
bucket = "c190312a4909453l14790296t1w301070396965-labbucket-flsgro8xdek1"
job_data_access_role = 'arn:aws:iam::...'

In [9]:
import json

with open('/home/ec2-user/SageMaker/en_us/capstone.ipynb', 'r') as f:
    nb = json.load(f)

# Search for bucket name in all cells
for cell in nb['cells']:
    src = ''.join(cell['source'])
    if 'bucket' in src.lower() and 'labbucket' in src.lower():
        print(src)
        print('---')

bucket = "c190312a4909453l14790296t1w670353558621-labbucket-zp0ornrldsqf"
job_data_access_role = 'arn:aws:iam::670353558621:role/service-role/c190312a4909453l14790296t1-ComprehendDataAccessRole-HO9gdiW5eAqD'
---
import boto3
import time
import json
import uuid

bucket = "c190312a4909453l14790296t1w301070396965-labbucket-flsgro8xdek1"
job_data_access_role = 'arn:aws:iam::301070396965:role/service-role/c190312a4909453l14790296t1-ComprehendDataAccessRole-gAlehpJYsV3j'

s3 = boto3.client('s3')
transcribe = boto3.client('transcribe')

print("✅ Clients ready")
---
import boto3

bucket = "c190312a4909453l14790296t1w301070396965-labbucket-flsgro8xdek1"
job_data_access_role = 'arn:aws:iam::670353558621:role/service-role/c190312a4909453l14790296t1-ComprehendDataAccessRole-gAlehpJYsV3j'

tests = {
    'Comprehend - DetectKeyPhrases': lambda: boto3.client('comprehend').detect_key_phrases(Text='test sentence', LanguageCode='en'),
    'Comprehend - DetectEntities': lambda: boto3.client('comprehend')

In [10]:
import boto3
import time
import json
import uuid

# CORRECT values for new lab session
bucket = "c190312a4909453l14790296t1w670353558621-labbucket-zp0ornrldsqf"
job_data_access_role = 'arn:aws:iam::670353558621:role/service-role/c190312a4909453l14790296t1-ComprehendDataAccessRole-HO9gdiW5eAqD'

s3 = boto3.client('s3')
transcribe = boto3.client('transcribe')

print(f"✅ Bucket: {bucket}")
print(f"✅ Role: {job_data_access_role}")

✅ Bucket: c190312a4909453l14790296t1w670353558621-labbucket-zp0ornrldsqf
✅ Role: arn:aws:iam::670353558621:role/service-role/c190312a4909453l14790296t1-ComprehendDataAccessRole-HO9gdiW5eAqD


In [11]:
# Test Transcribe
try:
    transcribe.get_transcription_job(TranscriptionJobName='test-reset')
except transcribe.exceptions.NotFoundException:
    print("✅ Transcribe accessible!")
except Exception as e:
    print(f"❌ Transcribe blocked: {e}")

# Test S3 with correct bucket
try:
    s3.list_objects_v2(Bucket=bucket)
    print("✅ S3 bucket accessible!")
except Exception as e:
    print(f"❌ S3 error: {e}")

❌ Transcribe blocked: An error occurred (AccessDeniedException) when calling the GetTranscriptionJob operation: User: arn:aws:sts::670353558621:assumed-role/c190312a4909453l14790296t1w6-SageMakerExecutionRole-O3GoCHvvnLyl/SageMaker is not authorized to perform: transcribe:GetTranscriptionJob on resource: arn:aws:transcribe:us-east-1:670353558621:transcription-job/test-reset because no identity-based policy allows the transcribe:GetTranscriptionJob action
✅ S3 bucket accessible!


In [12]:
import subprocess
result = subprocess.run('pip install SpeechRecognition pydub -q', shell=True, capture_output=True, text=True)
print("Libraries installed")

# Check if ffmpeg is available
result = subprocess.run('ffmpeg -version', shell=True, capture_output=True, text=True)
if result.returncode == 0:
    print("✅ ffmpeg available")
else:
    print("❌ ffmpeg not found, installing...")
    subprocess.run('conda install -y -c conda-forge ffmpeg -q', shell=True)
    print("✅ ffmpeg installed")

Libraries installed
❌ ffmpeg not found, installing...
Retrieving notices: ...working... done
Channels:
 - conda-forge
Platform: linux-64
Solving environment: ...working... done

## Package Plan ##

  environment location: /home/ec2-user/anaconda3/envs/python3

  added / updated specs:
    - ffmpeg


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    aom-3.9.1                  |       hac33072_0         2.6 MB  conda-forge
    dav1d-1.2.1                |       hd590300_0         742 KB  conda-forge
    ffmpeg-8.0.1               | gpl_hcddb375_914        11.9 MB  conda-forge
    glslang-16.2.0             |       h96af755_1         1.3 MB  conda-forge
    gmp-6.3.0                  |       hac33072_2         449 KB  conda-forge
    intel-gmmlib-22.10.0       |       hb700be7_0         990 KB  conda-forge
    intel-media-driver-26.1.6  |       hecca717_0         8.4 MB  conda-forge
    level

In [14]:
import os

# Download the smallest video first to test
test_video = 'Mod06_WrapUp.mp4'  # Only 19MB - smallest file
print(f"Downloading {test_video}...")

s3.download_file(
    'aws-tc-largeobjects',
    f'CUR-TF-200-ACMNLP-1/video/{test_video}',
    f'/tmp/{test_video}'
)

size = os.path.getsize(f'/tmp/{test_video}') / 1024 / 1024
print(f"✅ Downloaded {test_video}")
print(f"File size: {size:.1f} MB")


✅ Downloaded Mod06_WrapUp.mp4
File size: 18.9 MB


In [15]:
import subprocess
import speech_recognition as sr

# Step 1: Extract audio from video using ffmpeg
print("Extracting audio...")
result = subprocess.run([
    'ffmpeg', '-i', f'/tmp/{test_video}',
    '-ar', '16000',
    '-ac', '1',
    '-f', 'wav',
    '/tmp/test_audio.wav',
    '-y'
], capture_output=True, text=True)
print("✅ Audio extracted")

# Step 2: Transcribe using SpeechRecognition
print("Transcribing (this may take a minute)...")
r = sr.Recognizer()
with sr.AudioFile('/tmp/test_audio.wav') as source:
    audio = r.record(source)

try:
    text = r.recognize_google(audio)
    print("✅ Transcription successful!")
    print(f"Preview: {text[:300]}")
except Exception as e:
    print(f"❌ Error: {e}")

Extracting audio...
✅ Audio extracted
Transcribing (this may take a minute)...
✅ Transcription successful!
Preview: welcome back it's now time to review the module and wrap it up in summary in this module you learn how to describe the NLP use cases that are solved by using managed Amazon ml services and describe the managed ml services available for NLP good job thanks for watching we'll see you in the next modul


In [16]:
import os
import subprocess
import speech_recognition as sr
import json

# Get all video files
response = s3.list_objects_v2(Bucket='aws-tc-largeobjects', Prefix='CUR-TF-200-ACMNLP-1/video/')
video_files = [obj['Key'] for obj in response.get('Contents', []) if obj['Key'].endswith('.mp4')]
print(f"Found {len(video_files)} videos to transcribe\n")

transcripts = {}

for i, key in enumerate(video_files):
    filename = key.split('/')[-1]
    job_name = filename.replace('.mp4', '').replace(' ', '_')
    
    print(f"[{i+1}/{len(video_files)}] Processing: {filename}")
    
    try:
        # Download video
        s3.download_file('aws-tc-largeobjects', key, f'/tmp/video.mp4')
        
        # Extract audio
        subprocess.run([
            'ffmpeg', '-i', '/tmp/video.mp4',
            '-ar', '16000', '-ac', '1',
            '-f', 'wav', '/tmp/audio.wav', '-y'
        ], capture_output=True)
        
        # Transcribe in chunks (for long videos)
        r = sr.Recognizer()
        full_text = []
        
        with sr.AudioFile('/tmp/audio.wav') as source:
            duration = source.DURATION
            chunk_size = 60  # 60 second chunks
            offset = 0
            
            while offset < duration:
                audio_chunk = r.record(source, duration=min(chunk_size, duration - offset))
                try:
                    chunk_text = r.recognize_google(audio_chunk)
                    full_text.append(chunk_text)
                except sr.UnknownValueError:
                    pass  # Silent or unclear section
                except Exception as e:
                    print(f"  ⚠️ Chunk error at {offset}s: {e}")
                offset += chunk_size
        
        transcript = ' '.join(full_text)
        transcripts[job_name] = transcript
        
        # Save transcript to S3
        s3.put_object(
            Bucket=bucket,
            Key=f'transcripts/{job_name}.txt',
            Body=transcript.encode('utf-8')
        )
        print(f"  ✅ Done — {len(transcript)} characters")
        
        # Clean up temp files
        os.remove('/tmp/video.mp4')
        os.remove('/tmp/audio.wav')
        
    except Exception as e:
        print(f"  ❌ Failed: {e}")

print(f"\n🎉 Done! {len(transcripts)}/{len(video_files)} videos transcribed successfully")

Found 46 videos to transcribe

[1/46] Processing: Mod01_Course Overview.mp4
  ✅ Done — 9066 characters
[2/46] Processing: Mod02_Intro.mp4
  ✅ Done — 800 characters
[3/46] Processing: Mod02_Sect01.mp4
  ✅ Done — 6842 characters
[4/46] Processing: Mod02_Sect02.mp4
  ✅ Done — 9260 characters
[5/46] Processing: Mod02_Sect03.mp4
  ✅ Done — 7260 characters
[6/46] Processing: Mod02_Sect04.mp4
  ✅ Done — 5533 characters
[7/46] Processing: Mod02_Sect05.mp4
  ✅ Done — 2272 characters
[8/46] Processing: Mod02_WrapUp.mp4
  ✅ Done — 1061 characters
[9/46] Processing: Mod03_Intro.mp4
  ✅ Done — 1151 characters
[10/46] Processing: Mod03_Sect01.mp4
  ✅ Done — 5887 characters
[11/46] Processing: Mod03_Sect02_part1.mp4
  ✅ Done — 6962 characters
[12/46] Processing: Mod03_Sect02_part2.mp4
  ✅ Done — 3878 characters
[13/46] Processing: Mod03_Sect02_part3.mp4
  ✅ Done — 2612 characters
[14/46] Processing: Mod03_Sect03_part1.mp4
  ✅ Done — 3586 characters
[15/46] Processing: Mod03_Sect03_part2.mp4
  ✅ Done 

In [17]:
# Verify all transcripts in S3
response = s3.list_objects_v2(Bucket=bucket, Prefix='transcripts/')
transcript_files = [obj['Key'] for obj in response.get('Contents', [])]
print(f"✅ {len(transcript_files)} transcript files in S3:")
for f in transcript_files:
    print(f"  {f}")

✅ 46 transcript files in S3:
  transcripts/Mod01_Course_Overview.txt
  transcripts/Mod02_Intro.txt
  transcripts/Mod02_Sect01.txt
  transcripts/Mod02_Sect02.txt
  transcripts/Mod02_Sect03.txt
  transcripts/Mod02_Sect04.txt
  transcripts/Mod02_Sect05.txt
  transcripts/Mod02_WrapUp.txt
  transcripts/Mod03_Intro.txt
  transcripts/Mod03_Sect01.txt
  transcripts/Mod03_Sect02_part1.txt
  transcripts/Mod03_Sect02_part2.txt
  transcripts/Mod03_Sect02_part3.txt
  transcripts/Mod03_Sect03_part1.txt
  transcripts/Mod03_Sect03_part2.txt
  transcripts/Mod03_Sect03_part3.txt
  transcripts/Mod03_Sect04_part1.txt
  transcripts/Mod03_Sect04_part2.txt
  transcripts/Mod03_Sect04_part3.txt
  transcripts/Mod03_Sect05.txt
  transcripts/Mod03_Sect06.txt
  transcripts/Mod03_Sect07_part1.txt
  transcripts/Mod03_Sect07_part2.txt
  transcripts/Mod03_Sect07_part3.txt
  transcripts/Mod03_Sect08.txt
  transcripts/Mod03_WrapUp.txt
  transcripts/Mod04_Intro.txt
  transcripts/Mod04_Sect01.txt
  transcripts/Mod04_Sect0

In [18]:
import re
import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt_tab')

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

print("✅ NLTK ready")

[nltk_data] Downloading package punkt to /home/ec2-user/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/ec2-user/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/ec2-user/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package wordnet to /home/ec2-user/nltk_data...
[nltk_data] Downloading package omw-1.4 to /home/ec2-user/nltk_data...


✅ NLTK ready


[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/ec2-user/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [19]:
def normalize_text(text):
    # Lowercase
    text = text.lower()
    # Remove HTML tags
    text = re.compile('<.*?>').sub('', text)
    # Remove punctuation
    text = re.compile('[%s]' % re.escape('!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~')).sub(' ', text)
    # Remove extra whitespace
    text = re.sub('\s+', ' ', text).strip()
    # Tokenize and remove stopwords
    stop_words = set(stopwords.words('english'))
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
    # Lemmatize
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return ' '.join(tokens)

# Load, normalize and save all transcripts
normalized_transcripts = {}

response = s3.list_objects_v2(Bucket=bucket, Prefix='transcripts/')
transcript_files = [obj['Key'] for obj in response.get('Contents', [])]

print(f"Normalizing {len(transcript_files)} transcripts...\n")

for key in transcript_files:
    # Load from S3
    obj = s3.get_object(Bucket=bucket, Key=key)
    raw_text = obj['Body'].read().decode('utf-8')
    
    # Normalize
    normalized = normalize_text(raw_text)
    
    # Save normalized version
    filename = key.split('/')[-1]
    job_name = filename.replace('.txt', '')
    normalized_transcripts[job_name] = normalized
    
    s3.put_object(
        Bucket=bucket,
        Key=f'normalized/{filename}',
        Body=normalized.encode('utf-8')
    )
    print(f"  ✅ {filename} — {len(normalized)} chars")

print(f"\n✅ All {len(normalized_transcripts)} transcripts normalized!")

# Preview one
sample = list(normalized_transcripts.keys())[0]
print(f"\nSample ({sample}):")
print(normalized_transcripts[sample][:300])

<>:9: SyntaxWarning: invalid escape sequence '\s'
<>:9: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_18913/2991281509.py:9: SyntaxWarning: invalid escape sequence '\s'
  text = re.sub('\s+', ' ', text).strip()


Normalizing 46 transcripts...

  ✅ Mod01_Course_Overview.txt — 6463 chars
  ✅ Mod02_Intro.txt — 581 chars
  ✅ Mod02_Sect01.txt — 4656 chars
  ✅ Mod02_Sect02.txt — 6350 chars
  ✅ Mod02_Sect03.txt — 4463 chars
  ✅ Mod02_Sect04.txt — 4048 chars
  ✅ Mod02_Sect05.txt — 1518 chars
  ✅ Mod02_WrapUp.txt — 765 chars
  ✅ Mod03_Intro.txt — 863 chars
  ✅ Mod03_Sect01.txt — 3744 chars
  ✅ Mod03_Sect02_part1.txt — 4697 chars
  ✅ Mod03_Sect02_part2.txt — 2627 chars
  ✅ Mod03_Sect02_part3.txt — 1903 chars
  ✅ Mod03_Sect03_part1.txt — 2260 chars
  ✅ Mod03_Sect03_part2.txt — 3967 chars
  ✅ Mod03_Sect03_part3.txt — 1276 chars
  ✅ Mod03_Sect04_part1.txt — 2091 chars
  ✅ Mod03_Sect04_part2.txt — 2634 chars
  ✅ Mod03_Sect04_part3.txt — 3885 chars
  ✅ Mod03_Sect05.txt — 6362 chars
  ✅ Mod03_Sect06.txt — 3632 chars
  ✅ Mod03_Sect07_part1.txt — 1883 chars
  ✅ Mod03_Sect07_part2.txt — 1743 chars
  ✅ Mod03_Sect07_part3.txt — 4191 chars
  ✅ Mod03_Sect08.txt — 4706 chars
  ✅ Mod03_WrapUp.txt — 527 chars
  ✅ Mod04_

## 4. Extracting key phrases and topics
([Go to top](#Capstone-8:-Bringing-It-All-Together))

Use this section to extract the key phrases and topics from the videos.

In [21]:
import time

# Start Comprehend topic detection job
print("Starting Comprehend topic detection job...")

comprehend = boto3.client('comprehend')

response = comprehend.start_topics_detection_job(
    InputDataConfig={
        'S3Uri': f's3://{bucket}/normalized/',
        'InputFormat': 'ONE_DOC_PER_FILE'
    },
    OutputDataConfig={
        'S3Uri': f's3://{bucket}/topics/'
    },
    DataAccessRoleArn=job_data_access_role,
    NumberOfTopics=10
)

job_id = response['JobId']
print(f"✅ Job started! ID: {job_id}")

# Wait for job to complete
print("Waiting for job to complete (this takes 5-10 mins)...")
while True:
    status_response = comprehend.describe_topics_detection_job(JobId=job_id)
    status = status_response['TopicsDetectionJobProperties']['JobStatus']
    print(f"  Status: {status}")
    if status in ['COMPLETED', 'FAILED']:
        break
    time.sleep(30)

print(f"\n✅ Job {status}!")
if status == 'COMPLETED':
    output_uri = status_response['TopicsDetectionJobProperties']['OutputDataConfig']['S3Uri']
    print(f"Output at: {output_uri}")

Starting Comprehend topic detection job...
✅ Job started! ID: ebd95e81753d01b06cc565fa06309565
Waiting for job to complete (this takes 5-10 mins)...
  Status: SUBMITTED
  Status: IN_PROGRESS
  Status: IN_PROGRESS
  Status: IN_PROGRESS
  Status: IN_PROGRESS
  Status: IN_PROGRESS
  Status: IN_PROGRESS
  Status: IN_PROGRESS
  Status: IN_PROGRESS
  Status: IN_PROGRESS
  Status: IN_PROGRESS
  Status: IN_PROGRESS
  Status: IN_PROGRESS
  Status: IN_PROGRESS
  Status: IN_PROGRESS
  Status: IN_PROGRESS
  Status: IN_PROGRESS
  Status: IN_PROGRESS
  Status: IN_PROGRESS
  Status: IN_PROGRESS
  Status: IN_PROGRESS
  Status: COMPLETED

✅ Job COMPLETED!
Output at: s3://c190312a4909453l14790296t1w670353558621-labbucket-zp0ornrldsqf/topics/670353558621-TOPICS-ebd95e81753d01b06cc565fa06309565/output/output.tar.gz


In [22]:
import tarfile
import os

# Download and extract the output
output_key = f'topics/670353558621-TOPICS-ebd95e81753d01b06cc565fa06309565/output/output.tar.gz'

print("Downloading Comprehend output...")
s3.download_file(bucket, output_key, '/tmp/output.tar.gz')

# Extract tar file
with tarfile.open('/tmp/output.tar.gz', 'r:gz') as tar:
    tar.extractall('/tmp/comprehend_output/')

# List extracted files
files = os.listdir('/tmp/comprehend_output/')
print(f"✅ Extracted files: {files}")

✅ Extracted files: ['topic-terms.csv', 'doc-topics.csv']


/tmp/ipykernel_18913/3763974343.py:12: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall('/tmp/comprehend_output/')


In [23]:
import pandas as pd

# Load topic assignments
doc_topics = pd.read_csv('/tmp/comprehend_output/doc-topics.csv')
topic_terms = pd.read_csv('/tmp/comprehend_output/topic-terms.csv')

print("=== Topic Terms (what each topic is about) ===")
print(topic_terms.head(30))

print("\n=== Document Topics (which topic each video belongs to) ===")
print(doc_topics.head(20))

=== Topic Terms (what each topic is about) ===
    topic           term    weight
0       0          datum  0.122388
1       0           miss  0.037679
2       0         column  0.022657
3       0       variable  0.017615
4       0            set  0.023325
5       0        feature  0.018659
6       0         impute  0.010537
7       0           type  0.011956
8       0          panda  0.010038
9       0           drop  0.009357
10      1        feature  0.061780
11      1          datum  0.078480
12      1        outlier  0.021637
13      1          model  0.037739
14      1         column  0.016713
15      1         method  0.012410
16      1      numerical  0.011693
17      1         encode  0.010933
18      1   relationship  0.009378
19      1         filter  0.008287
20      2          datum  0.092296
21      2             aw  0.053463
22      2           glue  0.017653
23      2            etl  0.014422
24      2           exam  0.013568
25      2         secure  0.013816
26      

In [24]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import pandas as pd

# Build topic labels from top terms
topic_labels = {}
for topic_id in topic_terms['topic'].unique():
    terms = topic_terms[topic_terms['topic'] == topic_id].nlargest(3, 'weight')['term'].tolist()
    topic_labels[topic_id] = f"Topic {topic_id}: {', '.join(terms)}"

# Get primary topic per video (highest proportion)
primary_topics = doc_topics.sort_values('proportion', ascending=False)\
    .groupby('docname').first().reset_index()
primary_topics['video'] = primary_topics['docname'].str.replace('.txt', '', regex=False)
primary_topics['topic_label'] = primary_topics['topic'].map(topic_labels)

# Load all normalized transcripts for keyword search
print("Loading transcripts for search...")
all_transcripts = {}
response = s3.list_objects_v2(Bucket=bucket, Prefix='transcripts/')
for obj in response.get('Contents', []):
    key = obj['Key']
    name = key.split('/')[-1].replace('.txt', '')
    raw = s3.get_object(Bucket=bucket, Key=key)['Body'].read().decode('utf-8')
    all_transcripts[name] = raw

print(f"✅ Loaded {len(all_transcripts)} transcripts")
print(f"✅ Found {len(topic_labels)} topics")
print("\nTopic Labels:")
for k, v in topic_labels.items():
    print(f"  {v}")

Loading transcripts for search...
✅ Loaded 46 transcripts
✅ Found 10 topics

Topic Labels:
  Topic 0: datum, miss, set
  Topic 1: datum, feature, model
  Topic 2: datum, aw, glue
  Topic 3: label, image, model
  Topic 4: learn, machine, module
  Topic 5: learn, model, machine
  Topic 6: amazon, video, computer
  Topic 7: forecast, amazon, module
  Topic 8: cat, model, identify
  Topic 9: nlp, service, module


## 5. Creating the dashboard
([Go to top](#Capstone-8:-Bringing-It-All-Together))

Use this section to create the dashboard for your solution.

In [1]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

def render_results(results, keyword=''):
    # Fix: use len() instead of 'if not results' for DataFrames
    if len(results) == 0:
        display(HTML("<p style='color:gray'>No results found. Try a different keyword or topic.</p>"))
        return
    
    html = f"<p><b>{len(results)} video(s) found</b></p>"
    for _, row in results.iterrows():
        video = row['video']
        topic = row['topic_label']
        proportion = row['proportion']
        transcript = all_transcripts.get(video, '')
        
        # Highlight keyword in transcript preview
        preview = transcript[:300]
        if keyword:
            keyword_pos = transcript.lower().find(keyword.lower())
            if keyword_pos != -1:
                start = max(0, keyword_pos - 100)
                preview = '...' + transcript[start:keyword_pos+200] + '...'
                preview = preview.replace(
                    transcript[keyword_pos:keyword_pos+len(keyword)],
                    f"<mark><b>{transcript[keyword_pos:keyword_pos+len(keyword)]}</b></mark>"
                )
        
        html += f"""
        <div style="border:1px solid #ddd; border-radius:8px; padding:12px; 
                    margin:8px 0; background:#fafafa;">
            <div style="display:flex; justify-content:space-between; align-items:center;">
                <h3 style="margin:0; color:#232f3e">📹 {video.replace('_', ' ')}</h3>
                <span style="background:#ff9900; color:white; padding:3px 8px; 
                             border-radius:4px; font-size:12px;">
                    {topic} ({proportion:.0%})
                </span>
            </div>
            <p style="margin:8px 0 0 0; color:#555; font-size:13px;">{preview}</p>
        </div>
        """
    display(HTML(html))

def do_search(btn=None):
    keyword = search_box.value.strip()
    selected_topic = topic_dropdown.value
    
    with results_output:
        clear_output()
        filtered = primary_topics.copy()
        if selected_topic != 'All Topics':
            topic_id = int(selected_topic.split(':')[0].replace('Topic ', ''))
            filtered = filtered[filtered['topic'] == topic_id]
        if keyword:
            matching = [
                name for name, text in all_transcripts.items()
                if keyword.lower() in text.lower()
            ]
            filtered = filtered[filtered['video'].isin(matching)]
        render_results(filtered, keyword)

def do_clear(btn):
    search_box.value = ''
    topic_dropdown.value = 'All Topics'
    with results_output:
        clear_output()

search_btn.on_click(do_search)
clear_btn.on_click(do_clear)

display(stats_output)
display(widgets.HBox([search_box, topic_dropdown]))
display(widgets.HBox([search_btn, clear_btn]))
display(results_output)

# Show all on load
do_search()

NameError: name 'search_btn' is not defined

# Congratulations!

You have completed this lab, and you can now end the lab by following the lab guide instructions.

*©2023 Amazon Web Services, Inc. or its affiliates. All rights reserved. This work may not be reproduced or redistributed, in whole or in part, without prior written permission from Amazon Web Services, Inc. Commercial copying, lending, or selling is prohibited. All trademarks are the property of their owners.*
